# Chapter 4 — Task metrics, proxy metrics, and cost

Exact match and F1 on the date example, BLEU / ROUGE-L / F1 ranking a negation above a paraphrase,
calibration (ECE, Brier) under three amounts of overconfidence, and the Pareto front of five
configurations. Every cell **asserts** what it checks.

In [ ]:
import math, os, sys, random
sys.path.insert(0, os.path.abspath(".."))      # run from notebooks/
sys.path.insert(0, os.path.abspath("."))       # or from the repo root
import numpy as np
from evalkit import metrics as M

## 4.1 — exact match and F1

In [ ]:
GOLD = "The order ships on March 3."
for r in ["the order ships on March 3", "It ships March 3rd.", "Your order will ship on 3 March", "The order ships on March 13."]:
    print(f"{r:36s} EM {M.exact_match(r, GOLD, norm=False)}  normEM {M.exact_match(r, GOLD)}  F1 {M.token_f1(r, GOLD):.2f}")
assert round(M.token_f1("The order ships on March 13.", GOLD), 2) == 0.80     # the wrong date, high partial credit

## 4.2 — overlap prefers the negation

In [ ]:
REF = "You can return the item within 30 days for a full refund."
PARA = "Returns are accepted for a full refund if you send it back within a month."
NEG = "You cannot return the item within 30 days for a full refund."
for name, s in (("paraphrase (right)", PARA), ("negation (wrong)", NEG)):
    print(f"{name:20s} BLEU {M.sentence_bleu(s, [REF]):.3f}  ROUGE-L {M.rouge_l(s, REF):.3f}  F1 {M.token_f1(s, REF):.3f}")
assert M.sentence_bleu(NEG, [REF]) > 5 * M.sentence_bleu(PARA, [REF])
assert M.rouge_l(NEG, REF) > M.rouge_l(PARA, REF) and M.token_f1(NEG, REF) > M.token_f1(PARA, REF)

## 4.4 — calibration, and Exercise 4.3's threshold

In [ ]:
for oc in (0.0, 0.1, 0.2):
    c, o = M.simulate_confidences(5000, oc)
    print(f"overconfidence {oc:.1f}: accuracy {np.mean(o):.3f}  mean conf {np.mean(c):.3f}  ECE {M.ece(c, o) * 100:.1f}  Brier {M.brier(c, o):.3f}")
c, o = M.simulate_confidences(5000, 0.2)
shown = [y for x, y in zip(c, o) if x >= 0.9]
err = 1 - np.mean(shown)
print(f"policy 'show if >= 0.9' at +20 overconfidence: {len(shown)} shown, {err:.1%} of them wrong")
promised = 1 - np.mean([x for x in c if x >= 0.9])
print(f"stated confidence of shown answers averages {1 - promised:.3f}: a promised error rate of {promised:.1%}")
assert len(shown) == 3015 and round(err, 3) == 0.148 and err > 8 * promised
# recalibrate by bins on the first half, apply to the second half
half = 2500
bins = {}
for x, y in zip(c[:half], o[:half]):
    bins.setdefault(min(9, int(x * 10)), []).append(y)
rec = {b: np.mean(v) for b, v in bins.items()}
print("recalibrated accuracy per bin:", {b: round(v, 3) for b, v in sorted(rec.items())})
c2 = [rec.get(min(9, int(x * 10)), x) for x in c[half:]]
shown2 = [y for x, y in zip(c2, o[half:]) if x >= 0.9]
print(f"after recalibration: {len(shown2)} of {half} clear 0.9")
assert len(shown2) == 0 and round(max(rec.values()), 2) == 0.86     # nothing is 90% reliable

## 4.5 — the Pareto front and the cascade

In [ ]:
CONFIGS = {"large": (0.91, 0.0140), "medium": (0.88, 0.0042), "small": (0.79, 0.0009),
           "medium+retry": (0.90, 0.0061), "large-old": (0.87, 0.0150)}
print("front:", M.pareto_front(CONFIGS))
assert M.pareto_front(CONFIGS) == ["small", "medium", "medium+retry", "large"]
big, small = M.cost_per_call(1200, 300, 3, 15), M.cost_per_call(1200, 300, 0.25, 1.25)
casc = small + 0.22 * big
print(f"large ${big:.4f}  small ${small:.6f}  cascade ${casc:.5f}  monthly saving ${(big - casc) * 1e6:,.0f}")
assert round((big - casc) * 1e6) == 5643